# 🧠 Build Your Own Jev – Part 2: Choice (Pick One)

**Series:** [Part 1: Noul](https://www.kaggle.com/code/syedasimalishah/build-your-own-jev-part-1-noul-yes-no) | 👉 **Part 2: Choice** | [Part 3: Score](https://www.kaggle.com/code/syedasimalishah/build-your-own-jev-part-3-score-rate-on-scale)

## What you will learn
How to make a model **pick one option from your list**, and get a probability for every option.

## The data
The same 6 support tickets from an online shoe store.

## What happens in this notebook
1. **Setup:** install the libraries, load the model, build the engine, and load the tickets.
2. **Build `choice()`:** show the options as letters (A, B, C...) → softmax over the letters → the best option.
3. **Route tickets:** send each ticket to a team (shipping, returns, payments, technical, sales, other).
4. **Order bias test:** check if the model likes option "A" just because it is first, and fix it by shuffling.
5. **Route by confidence:** clear cases go automatically, unclear cases go to a human.

## Before you run
- ⚙️ Settings → **Accelerator: GPU** and **Internet: On**
- Then click **Run All**
- Look for `🔧 CHANGE` in the code. It shows what you can edit for your own use.

In [4]:
import json, math, random
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

# 🔧 CHANGE: MODEL_NAME
#   "Qwen/Qwen2.5-0.5B-Instruct" → faster, less smart (good for testing)
#   "Qwen/Qwen2.5-3B-Instruct"   → smarter, slower, needs more GPU memory
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"      # for batches: puts padding on the left, so every text ends at the last position
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🔧 CHANGE: dtype=torch.bfloat16 → uses half the memory (needs a newer GPU, like L4 or A100)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)
model.eval()
print("Loaded", MODEL_NAME, "on", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda


In [5]:
# ===== THE ENGINE: all 3 methods are built on these small functions =====

# 🔧 CHANGE: SYSTEM_PROMPT → tell the model its job and its world.
#   Example: "You are a senior support agent at an online shoe store." → better domain judgment
SYSTEM_PROMPT = "You are a careful, fair judge. You answer with one short label only."

def format_state(state):
    """Text stays text. A dict becomes neat JSON, so the model sees every field and its name."""
    if isinstance(state, (dict, list)):
        return json.dumps(state, indent=2, ensure_ascii=False)
    return str(state)

def build_prompt(state, task):
    """Joins STATE + TASK (the question and the allowed answers) into the model's chat format."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"STATE:\n{format_state(state)}\n\n{task}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# 🔧 CHANGE: batch_size → bigger = faster. Make it smaller if you see "CUDA out of memory".
@torch.no_grad()
def next_token_logits(prompts, batch_size=8):
    """Runs the model on many prompts. Returns one row of logits per prompt: [n_prompts, vocab_size]."""
    rows = []
    for i in range(0, len(prompts), batch_size):
        batch = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(device)
        out = model(**batch, logits_to_keep=1)   # keep only the LAST position (saves memory). Old transformers? Remove it.
        rows.append(out.logits[:, -1, :].float().cpu())
    return torch.cat(rows)

def token_ids(forms):
    """Token ids for each spelling, with and without a space.
    Example: ["Yes", "yes"] → ids of "Yes", " Yes", "yes", " yes"."""
    ids = set()
    for f in forms:
        for v in (f, " " + f):
            t = tokenizer.encode(v, add_special_tokens=False)
            if len(t) == 1:
                ids.add(t[0])
    if not ids:
        raise ValueError(f"None of {forms} is one token. Use short answers like letters or digits.")
    return sorted(ids)

def answer_logits(row, answers):
    """answers = {"yes": ["Yes", "yes"], "no": ["No", "no"]}
    Joins all spellings of one answer into ONE logit (logsumexp = adds their probabilities together)."""
    return torch.stack([torch.logsumexp(row[token_ids(forms)], dim=0) for forms in answers.values()])

def answer_probs(row, answers, temperature=1.0):
    """Keep only our answers → divide by temperature → softmax → probabilities that add up to 1."""
    return torch.softmax(answer_logits(row, answers) / temperature, dim=0)

def confidence(probs):
    """1.0 = all probability on one answer. 0.0 = spread evenly. Based on entropy."""
    if len(probs) < 2:
        return 1.0
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum().item()
    return 1 - entropy / math.log(len(probs))

print("Engine ready ✅")

Engine ready ✅


In [6]:
# Support tickets from an online shoe store.
# Each ticket is ONE STATE: everything the model needs to judge it.
# 🔧 CHANGE: put your own data here (emails, reviews, chats, CVs...).
#   Keep only the fields that help the decision. Extra fields = more noise and slower runs.
TICKETS = [
    {
        "ticket_id": "T-101",
        "customer": {"tier": "gold", "orders_last_year": 14},
        "order": {"item": "Trail running shoes", "price_usd": 120, "status": "delivered", "days_since_delivery": 3},
        "message": "Wrong size AGAIN. I ordered 40 and got 42, same as last month. I don't want another exchange, just give me my money back.",
    },
    {
        "ticket_id": "T-102",
        "customer": {"tier": "standard", "orders_last_year": 1},
        "order": {"item": "Leather boots", "price_usd": 210, "status": "in_transit", "days_since_order": 16},
        "message": "It's been 16 days. Tracking hasn't moved in a week. If it doesn't arrive by Friday I'm reporting you to consumer protection.",
    },
    {
        "ticket_id": "T-103",
        "customer": {"tier": "standard", "orders_last_year": 3},
        "order": None,
        "message": "Checkout keeps failing with error 502 when I press Pay. Tried two browsers. My card is 4532 7781 0092 4421 if that helps.",
    },
    {
        "ticket_id": "T-104",
        "customer": {"tier": "new", "orders_last_year": 0},
        "order": None,
        "message": "Hi! I run a running club with 30 members. Do you give a discount if we order 30 pairs together?",
    },
    {
        "ticket_id": "T-105",
        "customer": {"tier": "gold", "orders_last_year": 9},
        "order": {"item": "Sandals", "price_usd": 45, "status": "delivered", "days_since_delivery": 40},
        "message": "The sole came off after one month. Not happy. What can you do?",
    },
    {
        "ticket_id": "T-106",
        "customer": {"tier": "standard", "orders_last_year": 2},
        "order": {"item": "Sneakers", "price_usd": 85, "status": "delivered", "days_since_delivery": 1},
        "message": "I was charged twice for the same order, and the app crashes when I open my orders page.",
    },
]
print(len(TICKETS), "tickets loaded")

6 tickets loaded


In [7]:
def choice(states, instructions, criteria, temperature=1.0, n_orders=1, seed=0):
    """
    states   : list of states (text or dicts)
    criteria : {"label": "description", ...}  ← the options
    n_orders : try the options in N different orders and average the results.
               This fixes order bias, but is N times slower.
    returns  : one dict per state → choice, confidence, probabilities
    """
    labels = list(criteria)
    assert 2 <= len(labels) <= 26, "Use 2 to 26 options (the answer is one letter, A-Z)."

    # Order 1 = your order. The other orders are shuffled.
    rng = random.Random(seed)   # 🔧 CHANGE: seed → a different set of shuffles
    orders = [labels]
    for _ in range(n_orders - 1):
        shuffled = labels[:]
        rng.shuffle(shuffled)
        orders.append(shuffled)

    total = torch.zeros(len(states), len(labels))   # summed probabilities, kept in YOUR label order
    for order in orders:
        letters = [chr(65 + i) for i in range(len(order))]   # A, B, C, ...
        options = "\n".join(f"{L}. {lab}: {criteria[lab]}" for L, lab in zip(letters, order))
        task = (f"QUESTION: {instructions}\n"
                f"OPTIONS:\n{options}\n"
                f"Answer with only the letter of the best option.")
        rows = next_token_logits([build_prompt(s, task) for s in states])
        for i, row in enumerate(rows):
            probs = answer_probs(row, {L: [L] for L in letters}, temperature)
            for j, lab in enumerate(order):
                total[i, labels.index(lab)] += probs[j]   # map the letter back to its label
    total /= len(orders)

    results = []
    for row in total:
        probs = {lab: round(row[k].item(), 3) for k, lab in enumerate(labels)}
        results.append({
            "choice": max(probs, key=probs.get),
            "confidence": round(confidence(row), 2),
            "probabilities": probs,
        })
    return results

In [8]:
# CRITERIA = the options. Label → short description.
# 🔧 CHANGE: your own options. They should NOT overlap. Always keep an "other" option,
#            so the model is never forced to pick a wrong one.
TEAMS = {
    "shipping":  "Late, lost, or stuck deliveries and tracking problems",
    "returns":   "Returns, exchanges, refunds, and broken or wrong items",
    "payments":  "Double charges, failed payments, invoices",
    "technical": "Website or app errors, bugs, crashes",
    "sales":     "Prices, discounts, bulk or business orders",
    "other":     "Anything that does not fit the options above",
}
ROUTE_Q = "Which team should handle this ticket?"

# 🔧 CHANGE: n_orders=1 → 3x faster, but order bias can come back
results = choice(TICKETS, ROUTE_Q, TEAMS, n_orders=3)

pd.DataFrame([
    {"ticket_id": t["ticket_id"], "team": r["choice"], "confidence": r["confidence"], **r["probabilities"]}
    for t, r in zip(TICKETS, results)
])

,ticket_id,team,confidence,shipping,returns,payments,technical,sales,other
0,T-101,returns,1.00,0.000,1.000,0.000,0.000,0.0,0.000
1,T-102,shipping,1.00,1.000,0.000,0.000,0.000,0.0,0.000
2,T-103,payments,0.95,0.000,0.000,0.984,0.001,0.0,0.015
3,T-104,sales,1.00,0.000,0.000,0.000,0.000,1.0,0.000
4,T-105,returns,0.93,0.025,0.975,0.000,0.000,0.0,0.000
5,T-106,technical,0.47,0.001,0.000,0.193,0.610,0.0,0.196


In [9]:
# 🔬 Order bias test: T-106 has TWO problems (a payment and an app crash)
tricky = [TICKETS[5]]
print("1 order :", choice(tricky, ROUTE_Q, TEAMS, n_orders=1)[0]["probabilities"])
print("6 orders:", choice(tricky, ROUTE_Q, TEAMS, n_orders=6)[0]["probabilities"])

1 order : {'shipping': 0.0, 'returns': 0.0, 'payments': 0.011, 'technical': 0.989, 'sales': 0.0, 'other': 0.0}
6 orders: {'shipping': 0.0, 'returns': 0.0, 'payments': 0.097, 'technical': 0.805, 'sales': 0.0, 'other': 0.098}


In [10]:
# Route by confidence.
# 🔧 CHANGE: AUTO_ROUTE → higher = fewer wrong routes, but more tickets go to a human
AUTO_ROUTE = 0.75

for t, r in zip(TICKETS, results):
    top2 = sorted(r["probabilities"].items(), key=lambda kv: -kv[1])[:2]
    if r["confidence"] >= AUTO_ROUTE:
        print(f"{t['ticket_id']}: ✅ auto → {r['choice']}")
    else:
        print(f"{t['ticket_id']}: 🤔 human picks: {top2[0][0]} ({top2[0][1]:.2f}) or {top2[1][0]} ({top2[1][1]:.2f})")

T-101: ✅ auto → returns
T-102: ✅ auto → shipping
T-103: ✅ auto → payments
T-104: ✅ auto → sales
T-105: ✅ auto → returns
T-106: 🤔 human picks: technical (0.61) or other (0.20)


## 💡 Tips
- Does a ticket need **two teams** (like T-106)? Choice can only pick one.
  Use one **Noul per team** instead: "Does this ticket need the payments team?"
- Low confidence usually means the options overlap, or the state doesn't say enough.
- Do your options have an order (low → high)? Then use **Score**.